In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time
import pandas as pd
import warnings
import numpy as np
import logging
from tqdm import tqdm
from datetime import datetime

warnings.filterwarnings('ignore')
df = pd.read_csv('area_sales_community_df.csv', index_col=0)

In [12]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(f'scraper_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'), logging.StreamHandler()])
logger = logging.getLogger(__name__)

In [13]:
def get_geography_names(geography_name):
  all_names = geography_name.split(',')
  if len(all_names) == 3:
    return np.nan
  return ', '.join(all_names[:2])

df_pars = df[['geo_id', 'geographyName']]
df_pars[['neighborhood_city']] = df_pars['geographyName'].apply(lambda x: pd.Series(get_geography_names(x)))
df_pars = df_pars.drop('geographyName', axis=1)
df_pars['links'] = pd.Series()
df_pars = df_pars[~df_pars['neighborhood_city'].isna()].reset_index(drop=True)
df_pars

,geo_id,neighborhood_city,links
0,9cd7eab982c5c35bcd6f05d732d333c5,"Price End, North Port",NaN
1,b508f5dc4143a48154f0ff995b081678,"The Hills, Coral Springs",NaN
2,5b55d57472de1efb82e58cc003f53b78,"Whitesburg Estates, Huntsville",NaN
3,6a62dc1d0f1a59ae0c3de95037e1b683,"Kittyhawk, Dayton",NaN
4,8f3ec7cc1c199a0a566221bee1aeaa9c,"New Hampstead East, Savannah",NaN
...,...,...,...
4207,d1e3095ab678803f4b041f4235798872,"Flair, Lauderhill",NaN
4208,50ebe9038c385b6117d18e557d076841,"Lake Bonny, Lakeland",NaN
4209,3c969346ea2aa69d1b17362ecb262c76,"Bronx Park, St. Louis Park",NaN
4210,a6498e25964d71c4c3834d3f9c4bb11a,"Livingston, New York",NaN


In [16]:
options = Options()
driver = webdriver.Chrome(options=options)
url = 'https://www.greatschools.org/school-district-boundaries-map/'
driver.get(url)
driver.find_element(By.CSS_SELECTOR, '#privacy-banner-close').click()
driver.find_element(By.CSS_SELECTOR, '#cmpwrapper').shadow_root.find_element(By.CSS_SELECTOR, '#cmpwelcomebtnyes').click()

element = driver.find_element(By.CSS_SELECTOR, '[id^="ConnectedDistrictBoundaries-react-component"]')
id = element.get_attribute('id')

inp = driver.find_element(By.XPATH, f'//*[@id="{id}"]/div/div/div[1]/div[1]/div[1]/div[1]/input')
inp.clear()
inp.send_keys('Price End, North Port')
driver.find_element(By.XPATH, f'//*[@id="{id}"]/div/div/div[1]/div[1]/div[1]/div[2]').click()
WebDriverWait(driver, 60).until(EC.element_to_be_clickable((By.XPATH, f'//*[@id="{id}"]/div/div/div[2]/div[1]/div[3]/div[1]')))
driver.find_element(By.XPATH, f'//*[@id="{id}"]/div/div/div[2]/div[1]/div[3]/div[1]').click()
driver.find_element(By.XPATH, '/html/body/div[16]/div/div/div[2]/div[1]/div/div/div/div[2]/div/section/ul/div[2]/span/input').click()
driver.find_element(By.XPATH, '/html/body/div[16]/div/div/div[2]/div[2]/div/button').click()

In [32]:
def get_links(x):
    try:
        a = soup.find_all('div', class_='primary-section')[0].find_all('div', class_='header')
        ans = []
        if len(a) > 3:
            a = a[:3]
        for i in a:
            link = i.find_all('a', class_='name')[0]['href']
            ans.append(f'https://www.greatschools.org/{link}')
        return ans
    except:
        return np.nan

for i in df_pars[2275:].iterrows():
    logger.info(f'Парсинг района: {i[1]['neighborhood_city']}')
    inp = driver.find_element(By.XPATH, f'//*[@id="{id}"]/div/div/div[2]/div[1]/div[2]/div[1]/input')
    inp.clear()
    inp.send_keys(i[1]['neighborhood_city'])
    driver.find_element(By.XPATH, f'//*[@id="{id}"]/div/div/div[2]/div[1]/div[2]/div[2]').click()
    try:
        WebDriverWait(driver, 2).until(EC.alert_is_present())
        driver.switch_to.alert.accept()
        logger.warning(f'Район не найден: {i[1]['neighborhood_city']}')
    except:
        WebDriverWait(driver, 10).until(EC.invisibility_of_element_located((By.CSS_SELECTOR, '.spinny-wheel')))
        driver.find_element(By.XPATH, f'//*[@id="{id}"]/div/div/div[2]/div[1]/div[1]/div[1]').click()
        driver.find_element(By.CSS_SELECTOR, f'#{id} > div > div > div.primary-section > div.controls-section > div.filter-bar > div.sort-options > div > div > div > div > div').click()
        driver.find_element(By.CSS_SELECTOR, f'#{id} > div > div > div.primary-section > div.controls-section > div.filter-bar > div.sort-options > div > div > div > ul > li:nth-child(2)').click()
        WebDriverWait(driver, 10).until(EC.invisibility_of_element_located((By.CSS_SELECTOR, '.spinny-wheel')))
    
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        links = get_links(soup)
        df_pars['links'].iloc[i[0]] = links
        logger.info(f'Для района {i[1]['neighborhood_city']} получено {len(links) if links != np.nan else 0} ссылок на школы')
        logger.info(f'Прогресс - {i[0]}/{len(df_pars)}')

logger.info('КОНЕЦ')

2026-04-10 04:49:12,638 - INFO - Парсинг района: Port Jefferson Station,  Port Jefferson Station
2026-04-10 04:49:15,832 - INFO - Для района Port Jefferson Station,  Port Jefferson Station получено 3 ссылок на школы
2026-04-10 04:49:15,833 - INFO - Прогресс - 2275/4212
2026-04-10 04:49:15,834 - INFO - Парсинг района: Randall Manor,  New York
2026-04-10 04:49:18,967 - INFO - Для района Randall Manor,  New York получено 3 ссылок на школы
2026-04-10 04:49:18,968 - INFO - Прогресс - 2276/4212
2026-04-10 04:49:18,969 - INFO - Парсинг района: East Little York / Homestead,  Houston
2026-04-10 04:49:22,023 - INFO - Для района East Little York / Homestead,  Houston получено 3 ссылок на школы
2026-04-10 04:49:22,024 - INFO - Прогресс - 2277/4212
2026-04-10 04:49:22,025 - INFO - Парсинг района: Hunters Creek,  Hunters Creek
2026-04-10 04:49:25,169 - INFO - Для района Hunters Creek,  Hunters Creek получено 3 ссылок на школы
2026-04-10 04:49:25,170 - INFO - Прогресс - 2278/4212
2026-04-10 04:49:25,

In [84]:
df_pars = df_pars[~df_pars['links'].isna()].reset_index(drop=True)
df_pars = df_pars[df_pars['links'] != '[]'].reset_index(drop=True)
def link(x):
    return ' ; '.join(x[1:-1].replace("'", '').split(', '))
df_pars['links'] = df_pars['links'].apply(link)
df_pars

,geo_id,neighborhood_city,links
0,9cd7eab982c5c35bcd6f05d732d333c5,"Price End, North Port",https://www.greatschools.org//florida/north-po...
1,b508f5dc4143a48154f0ff995b081678,"The Hills, Coral Springs",https://www.greatschools.org//florida/coral-sp...
2,5b55d57472de1efb82e58cc003f53b78,"Whitesburg Estates, Huntsville",https://www.greatschools.org//alabama/huntsvil...
3,6a62dc1d0f1a59ae0c3de95037e1b683,"Kittyhawk, Dayton",https://www.greatschools.org//ohio/dayton/717-...
4,8f3ec7cc1c199a0a566221bee1aeaa9c,"New Hampstead East, Savannah",https://www.greatschools.org//georgia/savannah...
...,...,...,...
4181,d1e3095ab678803f4b041f4235798872,"Flair, Lauderhill",https://www.greatschools.org//florida/lauderda...
4182,50ebe9038c385b6117d18e557d076841,"Lake Bonny, Lakeland",https://www.greatschools.org//florida/lakeland...
4183,3c969346ea2aa69d1b17362ecb262c76,"Bronx Park, St. Louis Park",https://www.greatschools.org//minnesota/saint-...
4184,a6498e25964d71c4c3834d3f9c4bb11a,"Livingston, New York",https://www.greatschools.org//new-york/geneseo...


In [85]:
df_pars['links'].apply(lambda x: len(x.split(' ; '))).value_counts()

links
3    4145
2      27
1      14
Name: count, dtype: int64

In [86]:
df_pars.to_csv('school_links_by_neighborhoods.csv')